# Module 08: Model Evaluation

Deep-dive evaluation: Type I vs Type II errors, financial cost-utility functions, calibration curves, and K-Fold cross-validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score, brier_score_loss
)
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read dataset
try:
    credit_card_data = pd.read_csv("../creditcard.csv")
    print("Loaded ../creditcard.csv successfully.")
except FileNotFoundError:
    credit_card_data = pd.read_csv("creditcard.csv")
    print("Loaded creditcard.csv successfully.")

In [ ]:
# Preprocessing and Split
scaler = StandardScaler()
df = credit_card_data.copy()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df.drop(columns=['Time', 'Amount'], inplace=True)

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
print("Model ready for detailed evaluation.")

## Type I vs Type II Error Breakdown

In [ ]:
# Confusion Matrix with rates
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives  (Correct Normal):     {tn:,}")
print(f"False Positives (False Alarms):       {fp:,} (Type I error)")
print(f"False Negatives (Missed Frauds):      {fn:,} (Type II error)")
print(f"True Positives  (Detected Frauds):    {tp:,}")
print(f"Fraud Detection Sensitivity (Recall): {tp / (tp + fn):.2%}")
print(f"False Positive Rate:                  {fp / (fp + tn):.2%}")

## Business Cost-Utility Optimization Function

Assumptions:
- False Negative (Missed Fraud): Average loss ~$122
- False Positive (False Alarm): Verification call cost ~$5

In [ ]:
# Cost curve across decision thresholds
thresholds = np.linspace(0.05, 0.95, 50)
cost_fn = 122.0
cost_fp = 5.0

total_costs = []
fn_counts = []
fp_counts = []

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    c_matrix = confusion_matrix(y_test, preds)
    fn_val = c_matrix[1, 0]
    fp_val = c_matrix[0, 1]
    
    total_cost = (fn_val * cost_fn) + (fp_val * cost_fp)
    total_costs.append(total_cost)
    fn_counts.append(fn_val)
    fp_counts.append(fp_val)

optimal_idx = np.argmin(total_costs)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, total_costs, color='#d9534f', lw=2.5, label='Total Expected Financial Loss ($)')
plt.axvline(x=optimal_threshold, color='green', linestyle='--', label=f'Cost-Minimizing Threshold ({optimal_threshold:.2f})')
plt.title('Business Financial Impact vs Classification Threshold', fontsize=13, fontweight='bold')
plt.xlabel('Fraud Probability Threshold')
plt.ylabel('Expected Financial Loss ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(f"Minimum Total Cost: ${total_costs[optimal_idx]:,.2f}")
print(f"Missed Frauds at Optimal: {fn_counts[optimal_idx]} (out of {y_test.sum()})")
print(f"False Alarms at Optimal:  {fp_counts[optimal_idx]}")

## Stratified 5-Fold Cross-Validation

In [ ]:
# Stratified K-Fold CV to confirm model stability
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"5-Fold Cross-Validation ROC-AUC Scores: {cv_scores.round(4)}")
print(f"Mean ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Probability Calibration Curve

In [ ]:
# Check probability calibration
prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
brier_score = brier_score_loss(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, marker='o', lw=2, label=f'Logistic Regression (Brier: {brier_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
plt.title('Probability Calibration Curve', fontsize=12, fontweight='bold')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives (Actual Fraud)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()